# AQI Forecasting Project
## 02 — Data Cleaning

### Objective
Prepare a reliable hourly dataset for PM2.5-based forecasting.

### Cleaning principles
1. Never modify the original raw dataset.
2. Exclude stations with no PM2.5 observations.
3. Do not interpolate missing PM2.5 target values.
4. Investigate missing timestamps separately from missing pollutant values.
5. Do not remove extreme observations without evidence.
6. Preserve station and timestamp information.
7. Validate every cleaning operation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)

# Project paths
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_FILE = PROJECT_DIR.parent / "hourly_air_quality_raw.csv"
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

# Load fresh original data
raw_df = pd.read_csv(RAW_FILE, low_memory=False)

# Create working copy
clean_df = raw_df.copy()

# Convert timestamps
clean_df["observed_at"] = pd.to_datetime(
    clean_df["observed_at"],
    utc=True
)

clean_df["observed_at_ist"] = (
    clean_df["observed_at"].dt.tz_convert("Asia/Kolkata")
)

print("Original shape:", raw_df.shape)
print("Working shape:", clean_df.shape)

Original shape: (190514, 19)
Working shape: (190514, 20)


In [2]:
# Identify stations with at least one PM2.5 observation

eligible_stations = (
    clean_df.groupby("station_name")["pm25"]
    .count()
)

eligible_stations = eligible_stations[
    eligible_stations > 0
].index.tolist()

# Keep only eligible stations
clean_df = clean_df[
    clean_df["station_name"].isin(eligible_stations)
].copy()

print("Eligible stations:", len(eligible_stations))
print("Excluded stations:", 15 - len(eligible_stations))
print("Shape after station filtering:", clean_df.shape)

display(
    clean_df.groupby("station_name")
    .agg(
        Total_Rows=("pm25", "size"),
        Available_PM25=("pm25", "count"),
        Missing_PM25=("pm25", lambda x: x.isna().sum())
    )
)

Eligible stations: 13
Excluded stations: 2
Shape after station filtering: (179539, 20)


,Total_Rows,Available_PM25,Missing_PM25
station_name,,,
"Alipur, Delhi - DPCC",13506,13323,183
"Anand Vihar, New Delhi - DPCC",13736,13493,243
"Bawana, Delhi - DPCC",13894,13621,273
"Dwarka-Sector 8, Delhi - DPCC",13919,13750,169
"ITO, New Delhi - CPCB",13924,13626,298
"Mandir Marg, New Delhi - DPCC",13765,13113,652
"Mundka, Delhi - DPCC",13919,13744,175
"Patparganj, Delhi - DPCC",13784,13670,114
"Punjabi Bagh, Delhi - DPCC",13651,13386,265


# AQI Forecasting Project
## 03 — PM2.5 Missing Data and Gap Analysis

### Objective
Investigate missing PM2.5 readings across eligible monitoring stations.

### Questions
1. How many PM2.5 readings are missing?
2. How long are consecutive missing periods?
3. Which gaps are short enough to consider interpolation?
4. How does interpolation affect forecasting data?
5. Which missing values should remain unfilled?

### Important rule
Never modify the original dataset or fabricate PM2.5 target values.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = PROJECT_DIR.parent / "hourly_air_quality_raw.csv"

df = pd.read_csv(RAW_FILE, low_memory=False)

df["observed_at"] = pd.to_datetime(
    df["observed_at"], utc=True
)

df["observed_at_ist"] = (
    df["observed_at"].dt.tz_convert("Asia/Kolkata")
)

# Exclude stations with no PM2.5 readings
df = df[df["station_name"].isin(
    df.groupby("station_name")["pm25"].count()
      .loc[lambda x: x > 0].index
)].copy()

print("Dataset shape:", df.shape)
print("Stations:", df["station_name"].nunique())
print("Missing PM2.5 readings:", df["pm25"].isna().sum())

Dataset shape: (179539, 20)
Stations: 13
Missing PM2.5 readings: 3064


In [4]:
# Calculate consecutive missing PM2.5 gaps station-wise

gap_records = []

for station, group in df.groupby("station_name"):

    group = group.sort_values("observed_at_ist").copy()

    # Identify missing PM2.5 readings
    missing = group["pm25"].isna()

    # Create a new group whenever missing/available status changes
    group["gap_group"] = (missing != missing.shift()).cumsum()

    # Extract consecutive missing blocks
    missing_blocks = group[missing].groupby("gap_group")

    for _, block in missing_blocks:

        gap_records.append({
            "Station": station,
            "Gap_Start": block["observed_at_ist"].min(),
            "Gap_End": block["observed_at_ist"].max(),
            "Missing_Hours": len(block)
        })

pm25_gaps = pd.DataFrame(gap_records)

print("Total consecutive missing blocks:", len(pm25_gaps))
print("Total missing readings:", pm25_gaps["Missing_Hours"].sum())

display(
    pm25_gaps["Missing_Hours"]
    .value_counts()
    .sort_index()
    .rename_axis("Gap_Length_Hours")
    .reset_index(name="Number_of_Gaps")
)

Total consecutive missing blocks: 1300
Total missing readings: 3064


,Gap_Length_Hours,Number_of_Gaps
0,1,817
1,2,206
2,3,111
3,4,41
4,5,27
5,6,20
6,7,18
7,8,8
8,9,4
9,10,10


In [5]:
# Check whether missing PM2.5 blocks are truly consecutive hourly gaps

gap_time_validation = []

for station, group in df.groupby("station_name"):

    group = group.sort_values("observed_at_ist").copy()
    missing = group["pm25"].isna()

    group["gap_group"] = (missing != missing.shift()).cumsum()

    for _, block in group[missing].groupby("gap_group"):

        timestamps = block["observed_at_ist"].sort_values()

        if len(timestamps) > 1:
            intervals = timestamps.diff().dropna()
            is_hourly = (intervals == pd.Timedelta(hours=1)).all()
            max_interval = intervals.max()
        else:
            is_hourly = True
            max_interval = pd.Timedelta(0)

        gap_time_validation.append({
            "Station": station,
            "Gap_Start": timestamps.iloc[0],
            "Gap_End": timestamps.iloc[-1],
            "Missing_Rows": len(block),
            "Strictly_Hourly": is_hourly,
            "Maximum_Interval": max_interval
        })

gap_time_validation = pd.DataFrame(gap_time_validation)

print("Total blocks:", len(gap_time_validation))
print(
    "Strictly hourly blocks:",
    gap_time_validation["Strictly_Hourly"].sum()
)
print(
    "Blocks with timestamp interruptions:",
    (~gap_time_validation["Strictly_Hourly"]).sum()
)

display(
    gap_time_validation[
        ~gap_time_validation["Strictly_Hourly"]
    ].head(30)
)

Total blocks: 1300
Strictly hourly blocks: 1261
Blocks with timestamp interruptions: 39


,Station,Gap_Start,Gap_End,Missing_Rows,Strictly_Hourly,Maximum_Interval
8,"Alipur, Delhi - DPCC",2025-09-04 09:00:00+05:30,2025-09-04 13:00:00+05:30,3,False,0 days 03:00:00
33,"Alipur, Delhi - DPCC",2026-02-07 08:00:00+05:30,2026-02-07 17:00:00+05:30,2,False,0 days 09:00:00
40,"Alipur, Delhi - DPCC",2026-04-28 00:00:00+05:30,2026-04-28 04:00:00+05:30,2,False,0 days 04:00:00
67,"Alipur, Delhi - DPCC",2026-08-06 14:00:00+05:30,2026-08-06 19:00:00+05:30,4,False,0 days 03:00:00
89,"Alipur, Delhi - DPCC",2026-08-22 17:00:00+05:30,2026-08-22 20:00:00+05:30,3,False,0 days 02:00:00
121,"Anand Vihar, New Delhi - DPCC",2025-06-12 13:00:00+05:30,2025-06-12 15:00:00+05:30,2,False,0 days 02:00:00
172,"Anand Vihar, New Delhi - DPCC",2026-03-25 22:00:00+05:30,2026-03-26 10:00:00+05:30,12,False,0 days 02:00:00
179,"Anand Vihar, New Delhi - DPCC",2026-05-03 03:00:00+05:30,2026-05-03 13:00:00+05:30,3,False,0 days 09:00:00
372,"Dwarka-Sector 8, Delhi - DPCC",2025-08-22 17:00:00+05:30,2025-08-23 09:00:00+05:30,15,False,0 days 03:00:00
383,"Dwarka-Sector 8, Delhi - DPCC",2026-06-26 18:00:00+05:30,2026-06-26 23:00:00+05:30,4,False,0 days 03:00:00


## Complete Hourly Timeline

The raw dataset contains two types of missing PM2.5 information:

1. Existing observations where PM2.5 is null.
2. Missing hourly timestamps where no observation was recorded.

We will create a complete hourly timeline for each station
to identify both types.

No PM2.5 values will be imputed in this step.

In [7]:
# Create a complete hourly timeline for each eligible station

station_hourly_data = []

for station, group in df.groupby("station_name"):

    group = group.sort_values("observed_at_ist").copy()

    # Set timestamp as index
    group = group.set_index("observed_at_ist")

    # Create complete hourly index between first and last observation
    full_hourly_index = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="h"
    )

    # Reindex to expose completely missing timestamps
    hourly_group = group.reindex(full_hourly_index)

    hourly_group.index.name = "observed_at_ist"
    hourly_group["station_name"] = station

    station_hourly_data.append(hourly_group.reset_index())

# Combine all stations
hourly_df = pd.concat(
    station_hourly_data,
    ignore_index=True
)

# Identify PM2.5 availability
hourly_df["PM25_Missing"] = hourly_df["pm25"].isna()

print("Original eligible rows:", len(df))
print("Complete hourly rows:", len(hourly_df))

print(
    "Total unavailable PM2.5 hours:",
    hourly_df["PM25_Missing"].sum()
)

print(
    "Additional missing timestamps:",
    len(hourly_df) - len(df)
)

display(
    hourly_df.groupby("station_name")
    .agg(
        Expected_Hours=("pm25", "size"),
        Available_PM25=("pm25", "count"),
        Missing_PM25=("pm25", lambda x: x.isna().sum())
    )
    .sort_values("Missing_PM25", ascending=False)
)

Original eligible rows: 179539
Complete hourly rows: 197697
Total unavailable PM2.5 hours: 21222
Additional missing timestamps: 18158


,Expected_Hours,Available_PM25,Missing_PM25
station_name,,,
"Mandir Marg, New Delhi - DPCC",15209,13113,2096
"Alipur, Delhi - DPCC",15202,13323,1879
"Punjabi Bagh, Delhi - DPCC",15208,13386,1822
"R K Puram, Delhi - DPCC",15207,13489,1718
"Anand Vihar, New Delhi - DPCC",15206,13493,1713
"Vivek Vihar, Delhi - DPCC",15208,13549,1659
"Bawana, Delhi - DPCC",15206,13621,1585
"ITO, New Delhi - CPCB",15209,13626,1583
"Patparganj, Delhi - DPCC",15208,13670,1538


## Missing PM2.5: Timestamp Gaps vs Null Values

Separate unavailable PM2.5 hours into:

1. Missing timestamps: no observation row exists.
2. Null PM2.5: observation exists but its PM2.5 value is missing.

This distinction will guide our cleaning and imputation strategy.

In [8]:
# Compare original observations against the complete hourly timeline

original_station_summary = (
    df.groupby("station_name")
    .agg(
        Original_Rows=("observed_at_ist", "size"),
        Null_PM25=("pm25", lambda x: x.isna().sum()),
        Available_PM25=("pm25", "count")
    )
)

complete_station_summary = (
    hourly_df.groupby("station_name")
    .agg(
        Expected_Hours=("pm25", "size"),
        Total_Unavailable_PM25=("PM25_Missing", "sum")
    )
)

missing_breakdown = original_station_summary.join(
    complete_station_summary
)

missing_breakdown["Missing_Timestamps"] = (
    missing_breakdown["Expected_Hours"]
    - missing_breakdown["Original_Rows"]
)

missing_breakdown["Total_Unavailable_Check"] = (
    missing_breakdown["Missing_Timestamps"]
    + missing_breakdown["Null_PM25"]
)

missing_breakdown["Missing_Percentage"] = (
    missing_breakdown["Total_Unavailable_PM25"]
    / missing_breakdown["Expected_Hours"]
    * 100
)

display(
    missing_breakdown.sort_values(
        "Missing_Percentage",
        ascending=False
    ).round(2)
)

,Original_Rows,Null_PM25,Available_PM25,Expected_Hours,Total_Unavailable_PM25,Missing_Timestamps,Total_Unavailable_Check,Missing_Percentage
station_name,,,,,,,,
"Mandir Marg, New Delhi - DPCC",13765,652,13113,15209,2096,1444,2096,13.78
"Alipur, Delhi - DPCC",13506,183,13323,15202,1879,1696,1879,12.36
"Punjabi Bagh, Delhi - DPCC",13651,265,13386,15208,1822,1557,1822,11.98
"R K Puram, Delhi - DPCC",13727,238,13489,15207,1718,1480,1718,11.30
"Anand Vihar, New Delhi - DPCC",13736,243,13493,15206,1713,1470,1713,11.27
"Vivek Vihar, Delhi - DPCC",13769,220,13549,15208,1659,1439,1659,10.91
"Bawana, Delhi - DPCC",13894,273,13621,15206,1585,1312,1585,10.42
"ITO, New Delhi - CPCB",13924,298,13626,15209,1583,1285,1583,10.41
"Patparganj, Delhi - DPCC",13784,114,13670,15208,1538,1424,1538,10.11


## Predictor Missingness After Timeline Reconstruction

The complete hourly timeline contains both original observations
and timestamps that were absent from the raw dataset.

We will inspect missingness in every predictor variable to determine
which variables need imputation and which may need to be excluded.
PM2.5 will remain unimputed because it is our prediction target.

In [10]:
# Predictor variables (excluding PM2.5 target)

predictor_cols = [
    "co", "no", "no2", "nox", "o3",
    "pm10", "relative_humidity", "so2",
    "temperature", "wind_direction", "wind_speed"
]

# Missingness across the complete hourly timeline

predictor_missing = pd.DataFrame({
    "Missing_Count": hourly_df[predictor_cols].isna().sum(),
    "Missing_Percentage": (
        hourly_df[predictor_cols].isna().mean() * 100
    )
})

predictor_missing = predictor_missing.sort_values(
    "Missing_Percentage",
    ascending=False
)

display(predictor_missing.round(2))

,Missing_Count,Missing_Percentage
wind_direction,42502,21.50
so2,35877,18.15
wind_speed,35007,17.71
no2,33695,17.04
relative_humidity,33347,16.87
temperature,33300,16.84
o3,33026,16.71
co,21316,10.78
pm10,21089,10.67
no,20369,10.30


## Station-wise Predictor Missingness on Complete Timeline

Evaluate predictor availability for every station after hourly
timeline reconstruction.

This will help identify:
- Predictors that are unreliable at particular stations.
- Variables that can be retained.
- Variables requiring station-specific imputation.

In [11]:
# Missing percentage by station on complete hourly timeline

station_predictor_missing = (
    hourly_df.groupby("station_name")[predictor_cols]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

display(station_predictor_missing)

,co,no,no2,nox,o3,pm10,relative_humidity,so2,temperature,wind_direction,wind_speed
station_name,,,,,,,,,,,
"Alipur, Delhi - DPCC",12.72,11.80,11.91,11.67,12.70,12.62,14.68,12.45,14.00,14.02,23.80
"Anand Vihar, New Delhi - DPCC",11.45,12.08,100.00,10.65,86.85,12.45,10.15,100.00,10.11,10.25,10.17
"Bawana, Delhi - DPCC",10.82,10.88,10.26,10.19,9.15,9.89,8.87,10.50,8.87,9.79,11.16
"Dwarka-Sector 8, Delhi - DPCC",9.69,9.18,9.13,9.13,9.43,9.26,8.51,9.83,8.51,8.53,8.55
"ITO, New Delhi - CPCB",8.49,8.47,8.48,8.47,8.69,11.16,100.00,9.54,100.00,100.00,100.00
"Mandir Marg, New Delhi - DPCC",11.47,11.40,11.84,10.82,11.37,11.04,9.50,15.53,9.49,12.55,9.50
"Mundka, Delhi - DPCC",9.51,9.09,8.70,8.77,11.62,9.54,9.02,10.50,9.49,9.53,9.01
"Patparganj, Delhi - DPCC",11.23,10.46,10.97,11.00,11.90,9.96,10.32,10.41,10.32,12.33,10.34
"Punjabi Bagh, Delhi - DPCC",11.17,11.20,11.40,10.77,12.64,12.14,11.22,13.77,11.10,53.97,10.36


In [12]:
# Identify completely unavailable predictor combinations

unavailable_predictors = (
    station_predictor_missing
    .stack()
    .reset_index()
)

unavailable_predictors.columns = [
    "Station", "Variable", "Missing_Percentage"
]

unavailable_predictors = unavailable_predictors[
    unavailable_predictors["Missing_Percentage"] == 100
]

display(unavailable_predictors)

,Station,Variable,Missing_Percentage
13,"Anand Vihar, New Delhi - DPCC",no2,100.0
18,"Anand Vihar, New Delhi - DPCC",so2,100.0
50,"ITO, New Delhi - CPCB",relative_humidity,100.0
52,"ITO, New Delhi - CPCB",temperature,100.0
53,"ITO, New Delhi - CPCB",wind_direction,100.0
54,"ITO, New Delhi - CPCB",wind_speed,100.0


In [13]:
# Create a separate working copy. Never modify the raw dataframe.
cleaned_hourly_df = hourly_df.copy()

# Restore station metadata for hours inserted during timeline reconstruction.
metadata_cols = [
    "location_id",
    "candidate_station",
    "region",
    "latitude",
    "longitude"
]

station_metadata = (
    df.groupby("station_name")[metadata_cols]
    .first()
)

for col in metadata_cols:
    cleaned_hourly_df[col] = (
        cleaned_hourly_df["station_name"]
        .map(station_metadata[col])
    )

# Reconstruct the timestamp columns consistently.
cleaned_hourly_df["observed_at"] = (
    cleaned_hourly_df["observed_at_ist"].dt.tz_convert("UTC")
)

# Keep target missingness explicit.
cleaned_hourly_df["PM25_Missing"] = (
    cleaned_hourly_df["pm25"].isna()
)

# Sort and verify.
cleaned_hourly_df = (
    cleaned_hourly_df
    .sort_values(["station_name", "observed_at_ist"])
    .reset_index(drop=True)
)

print("Prepared dataset shape:", cleaned_hourly_df.shape)
print("Number of stations:", cleaned_hourly_df["station_name"].nunique())
print("Missing PM2.5 values:", cleaned_hourly_df["pm25"].isna().sum())
print("Duplicate station-hour records:",
      cleaned_hourly_df.duplicated(
          ["station_name", "observed_at_ist"]
      ).sum())

display(cleaned_hourly_df.head())

Prepared dataset shape: (197697, 21)
Number of stations: 13
Missing PM2.5 values: 21222
Duplicate station-hour records: 0


,observed_at_ist,observed_at,location_id,station_name,candidate_station,region,latitude,longitude,co,no,no2,nox,o3,pm10,pm25,relative_humidity,so2,temperature,wind_direction,wind_speed,PM25_Missing
0,2025-02-19 01:00:00+05:30,2025-02-18 19:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.150,5.0,61.8,0.0369,2.50,135.0,60.5,87.0,1.75,12.5,230.0,0.300,False
1,2025-02-19 02:00:00+05:30,2025-02-18 20:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.350,16.3,62.9,0.0466,1.15,142.0,59.8,88.8,1.90,12.3,226.0,0.400,False
2,2025-02-19 03:00:00+05:30,2025-02-18 21:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.645,23.5,62.1,0.0521,1.38,198.0,82.0,89.0,1.80,12.3,198.0,0.650,False
3,2025-02-19 04:00:00+05:30,2025-02-18 22:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.588,12.7,61.7,0.0431,1.43,262.0,146.0,88.8,2.27,12.2,213.0,0.750,False
4,2025-02-19 05:00:00+05:30,2025-02-18 23:30:00+00:00,6932,"Alipur, Delhi - DPCC",Alipur,North Delhi,28.815329,77.15301,0.563,11.5,60.5,0.0415,1.60,304.0,205.0,88.5,3.63,12.2,234.0,0.775,False


In [14]:
# Save reconstructed hourly dataset
OUTPUT_DIR = Path.cwd().parent / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "reconstructed_hourly_data.csv"

cleaned_hourly_df.to_csv(OUTPUT_FILE, index=False)

print("Saved successfully:", OUTPUT_FILE)
print("File size (MB):", round(OUTPUT_FILE.stat().st_size / (1024 * 1024), 2))

Saved successfully: C:\Users\anshu\Desktop\ML project\aqi-intelligence\aqi-intelligence\datanew\aqi_rebuild\data\reconstructed_hourly_data.csv
File size (MB): 35.14


In [15]:
# Missing-value summary for predictor variables
missing_summary = pd.DataFrame({
    "Missing_Count": cleaned_hourly_df[predictor_cols].isna().sum(),
    "Missing_Percentage": (
        cleaned_hourly_df[predictor_cols].isna().mean() * 100
    ).round(2)
})

display(missing_summary.sort_values(
    "Missing_Percentage", ascending=False
))

,Missing_Count,Missing_Percentage
wind_direction,42502,21.50
so2,35877,18.15
wind_speed,35007,17.71
no2,33695,17.04
relative_humidity,33347,16.87
temperature,33300,16.84
o3,33026,16.71
co,21316,10.78
pm10,21089,10.67
no,20369,10.30


In [16]:
# Work on a separate feature-preparation dataframe
feature_df = cleaned_hourly_df.copy()

# 1. Create missingness indicators for every predictor
for col in predictor_cols:
    feature_df[f"{col}_missing"] = feature_df[col].isna().astype("int8")

# 2. Convert wind direction into circular features
# 0° and 360° represent the same direction.
wind_radians = np.deg2rad(feature_df["wind_direction"] % 360)

feature_df["wind_direction_sin"] = np.sin(wind_radians)
feature_df["wind_direction_cos"] = np.cos(wind_radians)

# 3. Verify the new features
missing_flags = [f"{col}_missing" for col in predictor_cols]

print("Original shape:", cleaned_hourly_df.shape)
print("Feature dataset shape:", feature_df.shape)

print("\nMissingness indicator totals:")
display(feature_df[missing_flags].sum().to_frame("Missing_Count"))

print("\nWind feature examples:")
display(
    feature_df[
        ["wind_direction", "wind_direction_sin", "wind_direction_cos"]
    ].head(10)
)

Original shape: (197697, 21)
Feature dataset shape: (197697, 34)

Missingness indicator totals:


,Missing_Count
co_missing,21316
no_missing,20369
no2_missing,33695
nox_missing,19719
o3_missing,33026
pm10_missing,21089
relative_humidity_missing,33347
so2_missing,35877
temperature_missing,33300
wind_direction_missing,42502



Wind feature examples:


,wind_direction,wind_direction_sin,wind_direction_cos
0,230.0,-0.766044,-0.642788
1,226.0,-0.719340,-0.694658
2,198.0,-0.309017,-0.951057
3,213.0,-0.544639,-0.838671
4,234.0,-0.809017,-0.587785
5,240.0,-0.866025,-0.500000
6,234.0,-0.809017,-0.587785
7,248.0,-0.927184,-0.374607
8,247.0,-0.920505,-0.390731
9,267.0,-0.998630,-0.052336


In [17]:
# Sort chronologically within each station
feature_df = feature_df.sort_values(
    ["station_name", "observed_at_ist"]
).copy()

# Forecast PM2.5 exactly 1 hour and 3 hours ahead
feature_df["PM25_next_1h"] = (
    feature_df.groupby("station_name")["pm25"].shift(-1)
)

feature_df["PM25_next_3h"] = (
    feature_df.groupby("station_name")["pm25"].shift(-3)
)

# Verify target alignment for one station
sample_station = feature_df["station_name"].iloc[0]

sample = feature_df.loc[
    feature_df["station_name"] == sample_station,
    ["observed_at_ist", "pm25", "PM25_next_1h", "PM25_next_3h"]
].head(10)

display(sample)

print("\nTarget availability:")
print("1-hour target:", feature_df["PM25_next_1h"].notna().sum())
print("3-hour target:", feature_df["PM25_next_3h"].notna().sum())

,observed_at_ist,pm25,PM25_next_1h,PM25_next_3h
0,2025-02-19 01:00:00+05:30,60.5,59.8,146.0
1,2025-02-19 02:00:00+05:30,59.8,82.0,205.0
2,2025-02-19 03:00:00+05:30,82.0,146.0,174.0
3,2025-02-19 04:00:00+05:30,146.0,205.0,128.0
4,2025-02-19 05:00:00+05:30,205.0,174.0,119.0
5,2025-02-19 06:00:00+05:30,174.0,128.0,108.0
6,2025-02-19 07:00:00+05:30,128.0,119.0,90.8
7,2025-02-19 08:00:00+05:30,119.0,108.0,74.5
8,2025-02-19 09:00:00+05:30,108.0,90.8,59.3
9,2025-02-19 10:00:00+05:30,90.8,74.5,41.0



Target availability:
1-hour target: 176462
3-hour target: 176436


In [18]:
# Verify that shifted targets are exactly 1 and 3 hours ahead
check_df = feature_df[
    ["station_name", "observed_at_ist",
     "PM25_next_1h", "PM25_next_3h"]
].copy()

check_df["time_1h"] = (
    check_df.groupby("station_name")["observed_at_ist"].shift(-1)
)

check_df["time_3h"] = (
    check_df.groupby("station_name")["observed_at_ist"].shift(-3)
)

check_df["diff_1h"] = (
    check_df["time_1h"] - check_df["observed_at_ist"]
)

check_df["diff_3h"] = (
    check_df["time_3h"] - check_df["observed_at_ist"]
)

print("1-hour interval counts:")
display(check_df["diff_1h"].value_counts(dropna=False))

print("\n3-hour interval counts:")
display(check_df["diff_3h"].value_counts(dropna=False))

1-hour interval counts:


diff_1h
0 days 01:00:00    197684
NaT                    13
Name: count, dtype: int64


3-hour interval counts:


diff_3h
0 days 03:00:00    197658
NaT                    39
Name: count, dtype: int64

In [19]:
# Pollutants used in CPCB AQI calculation
aqi_pollutants = [
    "pm25", "pm10", "no2", "so2", "co", "o3"
]

# Count available pollutant readings per station-hour
feature_df["available_aqi_pollutants"] = (
    feature_df[aqi_pollutants].notna().sum(axis=1)
)

print("Available pollutant count distribution:")
display(
    feature_df["available_aqi_pollutants"]
    .value_counts()
    .sort_index()
    .to_frame("Station_Hours")
)

print("\nAverage available pollutants by station:")
display(
    feature_df.groupby("station_name")[
        "available_aqi_pollutants"
    ].mean().round(2).to_frame("Average_Pollutants")
)

Available pollutant count distribution:


,Station_Hours
available_aqi_pollutants,
0,18172
1,179
2,678
3,11795
4,3624
5,10953
6,152296



Average available pollutants by station:


,Average_Pollutants
station_name,
"Alipur, Delhi - DPCC",5.25
"Anand Vihar, New Delhi - DPCC",2.78
"Bawana, Delhi - DPCC",5.39
"Dwarka-Sector 8, Delhi - DPCC",5.43
"ITO, New Delhi - CPCB",5.43
"Mandir Marg, New Delhi - DPCC",5.25
"Mundka, Delhi - DPCC",5.40
"Patparganj, Delhi - DPCC",5.35
"Punjabi Bagh, Delhi - DPCC",5.27


In [20]:
# Inspect pollutant concentration ranges
pollutant_summary = feature_df[aqi_pollutants].describe().T

pollutant_summary = pollutant_summary[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
].round(3)

display(pollutant_summary)

print("\nData types:")
display(feature_df[aqi_pollutants].dtypes.to_frame("Data_Type"))

,count,mean,std,min,25%,50%,75%,max
pm25,176475.0,88.760,87.326,0.0,33.300,57.50,112.00,997.00
pm10,176608.0,197.808,140.296,0.0,99.000,163.00,259.00,1790.00
no2,164002.0,40.869,29.952,0.0,19.300,33.60,55.20,445.00
so2,161820.0,14.558,19.069,0.0,6.450,10.70,17.90,1910.00
co,176381.0,1.225,0.969,0.0,0.673,0.98,1.45,9.84
o3,164671.0,29.426,33.230,0.0,7.630,17.00,38.60,491.00



Data types:


,Data_Type
pm25,float64
pm10,float64
no2,float64
so2,float64
co,float64
o3,float64


In [21]:
import json
from pathlib import Path
import pandas as pd

CATALOG_PATH = (
    Path.cwd().parent.parent / "active_sensors_by_station.json"
)

with open(CATALOG_PATH, "r", encoding="utf-8") as f:
    catalog = json.load(f)

unit_records = []

for region, stations in catalog.items():
    for station, info in stations.items():
        for pollutant, sensor in info.get("active_sensors", {}).items():
            unit_records.append({
                "Station": info["station_name"],
                "Pollutant": pollutant,
                "Sensor_ID": sensor.get("sensor_id"),
                "Unit": sensor.get("units"),
                "Sensor_Name": sensor.get("name")
            })

units_df = pd.DataFrame(unit_records)

unit_summary = (
    units_df.groupby(["Pollutant", "Unit"])
    .size()
    .reset_index(name="Station_Count")
)

display(unit_summary.sort_values("Pollutant"))

print("Detailed station-wise units:")
display(
    units_df.pivot(
        index="Station",
        columns="Pollutant",
        values="Unit"
    )
)

,Pollutant,Unit,Station_Count
0,co,ppb,25
1,no,ppb,25
2,no2,ppb,25
3,nox,ppb,25
4,o3,µg/m³,25
5,pm10,µg/m³,25
6,pm25,µg/m³,25
7,relativehumidity,%,22
8,so2,ppb,23
9,temperature,c,22


Detailed station-wise units:


Pollutant,co,no,no2,nox,o3,pm10,pm25,relativehumidity,so2,temperature,wind_direction,wind_speed
Station,,,,,,,,,,,,
"Alipur, Delhi - DPCC",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,%,ppb,c,deg,m/s
"Anand Vihar, New Delhi - DPCC",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,%,ppb,c,deg,m/s
"Ashok Vihar, Delhi - DPCC",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,%,ppb,c,deg,m/s
"Aya Nagar, New Delhi - IMD",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,NaN,NaN,NaN,NaN,NaN
"Bawana, Delhi - DPCC",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,%,ppb,c,deg,m/s
"Dr. Karni Singh Shooting Range, Delhi - DPCC",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,%,ppb,c,deg,m/s
"Dwarka-Sector 8, Delhi - DPCC",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,%,ppb,c,deg,m/s
"ITO, New Delhi - CPCB",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,NaN,ppb,NaN,NaN,NaN
"Jahangirpuri, Delhi - DPCC",ppb,ppb,ppb,ppb,µg/m³,µg/m³,µg/m³,%,ppb,c,deg,m/s


In [22]:

# Calculate trailing 24-hour PM2.5 average
# Require at least 16 valid hourly readings.

feature_df = feature_df.sort_values(
    ["station_name", "observed_at_ist"]
).copy()

feature_df["pm25_24h_avg"] = (
    feature_df.groupby("station_name")["pm25"]
    .transform(
        lambda s: s.rolling(
            window=24,
            min_periods=16
        ).mean()
    )
)

print("Valid 24-hour PM2.5 averages:")
print(feature_df["pm25_24h_avg"].notna().sum())

print("\nUnavailable averages:")
print(feature_df["pm25_24h_avg"].isna().sum())

print("\nSummary:")
display(feature_df["pm25_24h_avg"].describe().round(2))

print("\nExample:")
display(
    feature_df[
        ["station_name", "observed_at_ist", "pm25",
         "pm25_24h_avg"]
    ].head(30)
)


Valid 24-hour PM2.5 averages:
177017

Unavailable averages:
20680

Summary:


count    177017.00
mean         88.69
std          75.37
min           0.00
25%          38.13
50%          62.26
75%         109.54
max         621.44
Name: pm25_24h_avg, dtype: float64


Example:


,station_name,observed_at_ist,pm25,pm25_24h_avg
0,"Alipur, Delhi - DPCC",2025-02-19 01:00:00+05:30,60.5,NaN
1,"Alipur, Delhi - DPCC",2025-02-19 02:00:00+05:30,59.8,NaN
2,"Alipur, Delhi - DPCC",2025-02-19 03:00:00+05:30,82.0,NaN
3,"Alipur, Delhi - DPCC",2025-02-19 04:00:00+05:30,146.0,NaN
4,"Alipur, Delhi - DPCC",2025-02-19 05:00:00+05:30,205.0,NaN
5,"Alipur, Delhi - DPCC",2025-02-19 06:00:00+05:30,174.0,NaN
6,"Alipur, Delhi - DPCC",2025-02-19 07:00:00+05:30,128.0,NaN
7,"Alipur, Delhi - DPCC",2025-02-19 08:00:00+05:30,119.0,NaN
8,"Alipur, Delhi - DPCC",2025-02-19 09:00:00+05:30,108.0,NaN
9,"Alipur, Delhi - DPCC",2025-02-19 10:00:00+05:30,90.8,NaN


In [23]:

# Step 1: Convert PM2.5 rolling averages into categories

conditions = [
    feature_df["pm25_24h_avg"] <= 30,
    feature_df["pm25_24h_avg"] <= 60,
    feature_df["pm25_24h_avg"] <= 90,
    feature_df["pm25_24h_avg"] <= 120,
    feature_df["pm25_24h_avg"] <= 250,
    feature_df["pm25_24h_avg"] > 250
]

categories = [
    "Good",
    "Satisfactory",
    "Moderate",
    "Poor",
    "Very Poor",
    "Severe"
]

feature_df["pm25_aqi_category"] = np.select(
    conditions,
    categories,
    default="Unavailable"
)

# Preserve unavailable rolling averages as missing labels
feature_df.loc[
    feature_df["pm25_24h_avg"].isna(),
    "pm25_aqi_category"
] = pd.NA

# Step 2: Create future classification targets
feature_df["target_1h"] = (
    feature_df.groupby("station_name")[
        "pm25_aqi_category"
    ].shift(-1)
)

feature_df["target_3h"] = (
    feature_df.groupby("station_name")[
        "pm25_aqi_category"
    ].shift(-3)
)

# Step 3: Inspect category distribution
print("Current PM2.5 category distribution:")
display(
    feature_df["pm25_aqi_category"]
    .value_counts()
    .reindex(categories)
    .to_frame("Count")
)

print("\nFuture target availability:")
print("1-hour:", feature_df["target_1h"].notna().sum())
print("3-hour:", feature_df["target_3h"].notna().sum())

print("\nExample:")
display(
    feature_df[
        ["station_name", "observed_at_ist",
         "pm25_24h_avg", "pm25_aqi_category",
         "target_1h", "target_3h"]
    ].head(30)
)


Current PM2.5 category distribution:


,Count
pm25_aqi_category,
Good,25099
Satisfactory,60098
Moderate,34446
Poor,18700
Very Poor,28322
Severe,10352



Future target availability:
1-hour: 177017
3-hour: 177017

Example:


,station_name,observed_at_ist,pm25_24h_avg,pm25_aqi_category,target_1h,target_3h
0,"Alipur, Delhi - DPCC",2025-02-19 01:00:00+05:30,NaN,NaN,NaN,NaN
1,"Alipur, Delhi - DPCC",2025-02-19 02:00:00+05:30,NaN,NaN,NaN,NaN
2,"Alipur, Delhi - DPCC",2025-02-19 03:00:00+05:30,NaN,NaN,NaN,NaN
3,"Alipur, Delhi - DPCC",2025-02-19 04:00:00+05:30,NaN,NaN,NaN,NaN
4,"Alipur, Delhi - DPCC",2025-02-19 05:00:00+05:30,NaN,NaN,NaN,NaN
5,"Alipur, Delhi - DPCC",2025-02-19 06:00:00+05:30,NaN,NaN,NaN,NaN
6,"Alipur, Delhi - DPCC",2025-02-19 07:00:00+05:30,NaN,NaN,NaN,NaN
7,"Alipur, Delhi - DPCC",2025-02-19 08:00:00+05:30,NaN,NaN,NaN,NaN
8,"Alipur, Delhi - DPCC",2025-02-19 09:00:00+05:30,NaN,NaN,NaN,NaN
9,"Alipur, Delhi - DPCC",2025-02-19 10:00:00+05:30,NaN,NaN,NaN,NaN


In [24]:
# Compare current and future target distributions

print("1-hour target distribution:")
display(
    feature_df["target_1h"]
    .value_counts()
    .reindex(categories, fill_value=0)
    .to_frame("Count")
)

print("\n3-hour target distribution:")
display(
    feature_df["target_3h"]
    .value_counts()
    .reindex(categories, fill_value=0)
    .to_frame("Count")
)

print("\nTarget availability by station:")
display(
    feature_df.groupby("station_name")[
        ["pm25_aqi_category", "target_1h", "target_3h"]
    ].count()
)

1-hour target distribution:


,Count
target_1h,
Good,25099
Satisfactory,60098
Moderate,34446
Poor,18700
Very Poor,28322
Severe,10352



3-hour target distribution:


,Count
target_3h,
Good,25099
Satisfactory,60098
Moderate,34446
Poor,18700
Very Poor,28322
Severe,10352



Target availability by station:


,pm25_aqi_category,target_1h,target_3h
station_name,,,
"Alipur, Delhi - DPCC",13363,13363,13363
"Anand Vihar, New Delhi - DPCC",13539,13539,13539
"Bawana, Delhi - DPCC",13722,13722,13722
"Dwarka-Sector 8, Delhi - DPCC",13844,13844,13844
"ITO, New Delhi - CPCB",13643,13643,13643
"Mandir Marg, New Delhi - DPCC",13125,13125,13125
"Mundka, Delhi - DPCC",13832,13832,13832
"Patparganj, Delhi - DPCC",13665,13665,13665
"Punjabi Bagh, Delhi - DPCC",13496,13496,13496


In [25]:
# Check whether future labels actually change
print("Current vs 1-hour target:")
print(
    "Different labels:",
    (feature_df["pm25_aqi_category"] != feature_df["target_1h"])
    .sum()
)

print("\nCurrent vs 3-hour target:")
print(
    "Different labels:",
    (feature_df["pm25_aqi_category"] != feature_df["target_3h"])
    .sum()
)

# Show actual transitions
display(
    feature_df.loc[
        feature_df["pm25_aqi_category"].notna()
        & feature_df["target_3h"].notna()
        & (feature_df["pm25_aqi_category"] != feature_df["target_3h"]),
        ["station_name", "observed_at_ist",
         "pm25_aqi_category", "target_3h"]
    ].head(15)
)

Current vs 1-hour target:
Different labels: 27153

Current vs 3-hour target:
Different labels: 38541


,station_name,observed_at_ist,pm25_aqi_category,target_3h
15,"Alipur, Delhi - DPCC",2025-02-19 16:00:00+05:30,Poor,Moderate
16,"Alipur, Delhi - DPCC",2025-02-19 17:00:00+05:30,Poor,Moderate
19,"Alipur, Delhi - DPCC",2025-02-19 20:00:00+05:30,Moderate,Poor
20,"Alipur, Delhi - DPCC",2025-02-19 21:00:00+05:30,Moderate,Poor
21,"Alipur, Delhi - DPCC",2025-02-19 22:00:00+05:30,Moderate,Poor
26,"Alipur, Delhi - DPCC",2025-02-20 03:00:00+05:30,Poor,Moderate
27,"Alipur, Delhi - DPCC",2025-02-20 04:00:00+05:30,Poor,Moderate
28,"Alipur, Delhi - DPCC",2025-02-20 05:00:00+05:30,Poor,Moderate
30,"Alipur, Delhi - DPCC",2025-02-20 07:00:00+05:30,Moderate,Poor
31,"Alipur, Delhi - DPCC",2025-02-20 08:00:00+05:30,Moderate,Poor


In [26]:
# Save the completed feature and target dataset

OUTPUT_FILE = OUTPUT_DIR / "pm25_forecasting_dataset.csv"

feature_df.to_csv(OUTPUT_FILE, index=False)

print("Dataset saved:", OUTPUT_FILE)
print("Shape:", feature_df.shape)
print("Size (MB):", round(
    OUTPUT_FILE.stat().st_size / (1024 * 1024), 2
))

Dataset saved: C:\Users\anshu\Desktop\ML project\aqi-intelligence\aqi-intelligence\datanew\aqi_rebuild\data\pm25_forecasting_dataset.csv
Shape: (197697, 41)
Size (MB): 54.98
